In [ ]:
import torch
import torch.nn as nn
import torchaudio
from transformers import WhisperProcessor, WhisperModel, AutoTokenizer, AutoModelForCausalLM

# -----------------------------
# 1. Load Whisper (frozen)
# -----------------------------
whisper_model_name = "openai/whisper-large-v2"
processor = WhisperProcessor.from_pretrained(whisper_model_name)
whisper_model = WhisperModel.from_pretrained(whisper_model_name)
whisper_model.eval()  # Freeze Whisper
for param in whisper_model.parameters():
    param.requires_grad = False

# -----------------------------
# 2. Load LLM (frozen)
# -----------------------------
llm_model_name = "Qwen/Qwen-Audio-Chat"  # Replace with actual LLM
tokenizer = AutoTokenizer.from_pretrained(llm_model_name,)
llm = AutoModelForCausalLM.from_pretrained(llm_model_name)
llm.eval()  # Freeze LLM
for param in llm.parameters():
    param.requires_grad = False

# -----------------------------
# 3. Projection layer
# -----------------------------
whisper_hidden_size = whisper_model.config.d_model
llm_hidden_size = llm.config.hidden_size
projection = nn.Linear(whisper_hidden_size, llm_hidden_size)

# -----------------------------
# 4. Forward function
# -----------------------------
def forward(audio_path, instruction_prompt, label_token="Fake", device="cpu"):
    # Load audio
    waveform, sr = torchaudio.load(audio_path)

    # Convert stereo to mono if needed
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0)

    # Resample to 16 kHz
    waveform = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)(waveform)

    # Convert to numpy float32
    waveform_np = waveform.numpy().astype("float32").flatten()  # shape: (n_samples,)

    # Convert to input features
    input_features = processor(waveform_np, sampling_rate=16000, return_tensors="pt").input_features.to(device)

    # Forward through Whisper to get frame embeddings
    with torch.no_grad():
        whisper_outputs = whisper_model(input_features)
        frame_embeddings = whisper_outputs.last_hidden_state  # (1, seq_len, whisper_hidden_size)

    # Project embeddings to LLM embedding space
    projected_embeddings = projection(frame_embeddings.to(device))  # (1, seq_len, llm_hidden_size)

    # Tokenize instruction prompt
    prompt_ids = tokenizer(instruction_prompt, return_tensors="pt").input_ids.to(device)
    prompt_embeddings = llm.get_input_embeddings()(prompt_ids)

    # Concatenate projected audio embeddings with prompt embeddings
    llm_input_embeddings = torch.cat([projected_embeddings, prompt_embeddings], dim=1)

    # Forward through LLM
    outputs = llm(inputs_embeds=llm_input_embeddings)
    logits = outputs.logits  # (1, seq_len_total, vocab_size)

    # Compute cross-entropy loss with target token
    target_ids = tokenizer(label_token, return_tensors="pt").input_ids.to(device)
    loss_fn = nn.CrossEntropyLoss()
    loss = loss_fn(logits[:, -1, :], target_ids[:, 0])

    return loss, logits

# -----------------------------
# 5. Example usage
# -----------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
projection.to(device)
llm.to(device)

audio_path = "Recording.wav"
instruction_prompt = "Is this audio fake or real?"

loss, logits = forward(audio_path, instruction_prompt, label_token="Fake", device=device)
print("Loss:", loss.item())


/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


The repository for Qwen/Qwen-Audio-Chat contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/Qwen/Qwen-Audio-Chat.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


audio_start_id: 155163, audio_end_id: 155164, audio_pad_id: 151851.


/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


The repository for Qwen/Qwen-Audio-Chat contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/Qwen/Qwen-Audio-Chat.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y
The repository for Qwen/Qwen-Audio-Chat contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/Qwen/Qwen-Audio-Chat.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N]  y


model-00006-of-00009.safetensors:   0%|          | 0.00/1.93G [00:00<?, ?B/s]

model-00007-of-00009.safetensors:   0%|          | 0.00/1.93G [00:00<?, ?B/s]

model-00008-of-00009.safetensors:   0%|          | 0.00/1.87G [00:00<?, ?B/s]

In [ ]:
#Project frame embeddings from Whisper to LLM token input
projection = nn.Linear(whisper_hidden_size, llm_hidden_size)
llm_input_embeddings = projection(H_a)


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2-1.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_4bit=True,
)


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


...

The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

/home/chibu/.venvs/tensorflow/lib/python3.12/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

ValueError: `.to` is not supported for `4-bit` or `8-bit` bitsandbytes models. Please use the model as it is, since the model has already been set to the correct devices and casted to the correct `dtype`.

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "Qwen/Qwen2-1.5B"

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    load_in_4bit=True,           # Quantized → don't use device_map
)
model.eval()
model = model.cuda()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
`low_cpu_mem_usage` was None, now set to True since model is quantized.


ValueError: `.to` is not supported for `4-bit` or `8-bit` bitsandbytes models. Please use the model as it is, since the model has already been set to the correct devices and casted to the correct `dtype`.

In [9]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "Qwen/Qwen2.5-0.5B" # Example Qwen model

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Load model directly to GPU using device_map="auto"
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto"  # <--- This replaces model.cuda()
)

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# No need for model.cuda(), the model is already there!
print(f"Model loaded on: {model.device}")

config.json:   0%|          | 0.00/681 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/138 [00:00<?, ?B/s]

ValueError: `.to` is not supported for `4-bit` or `8-bit` bitsandbytes models. Please use the model as it is, since the model has already been set to the correct devices and casted to the correct `dtype`.

In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2-1.5B"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    load_in_8bit_fp32_cpu_offload=True,  # handles overflow to CPU
)

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

# Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    trust_remote_code=True,
    device_map="auto"  # automatically puts what fits on GPU, rest to CPU
)

# ✅ DO NOT do: model.cuda() or model.to("cuda")
print("Model loaded successfully on 4-bit with auto device mapping.")


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


ValueError: `.to` is not supported for `4-bit` or `8-bit` bitsandbytes models. Please use the model as it is, since the model has already been set to the correct devices and casted to the correct `dtype`.

In [13]:
!pip install transformers==4.34.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.5/121.5 kB 1.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 1.6 MB/s eta 0:00:0000:0100:010m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 1.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.0/295.0 kB 2.0 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.36.0
    Uninstalling huggingface-hub-0.36.0:
      Successfully uninstalled huggingface-hub-0.36.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.15.2
    Uninstalling tokenizers-0.15.2:
      Successfully uninstalled tokenizers-0.15.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.18.0 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.17.3 which is incompatible.
accel

NameError: name 'load_protocol' is not defined